# finance / portfolio / profile

**Intent:** For an input share held in risky assets (`finance_portfolio_profile_risky_share_pct`, percent) and an input withdrawal rate (`finance_portfolio_profile_withdrawal_rate_pct`, percent of initial wealth, constant in real terms), output the expected return (`finance_portfolio_profile_return_pct`) and the volatility (`finance_portfolio_profile_volatility_pct`) of the mix, both in percent a year, and the probability that the portfolio runs out of money within 30 years (`finance_portfolio_profile_shortfall_probability_pct`, percent), so any mix and withdrawal can be compared on risk, return and the chance of a shortfall.

**Notes:**
- Placeholder: two assets and their correlation are recovered from synthetic points, and the shortfall probability is a logistic curve fitted to a fixed seed simulation of the mix
- Data splits: `training` thirty share and withdrawal combinations with the return, volatility and simulated shortfall probability; `validation` twenty further; `backtest` twenty across the range, held back for acceptance; `golden` the exact combinations that become `golden.json`
- One mix: the shortfall simulation uses the return and volatility that the same model gives for the share, so they cannot disagree. The returns are real

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env` and the signature, and declare the model name, target and dataset names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/portfolio/profile"
MODEL_NAME = "finance_portfolio_profile"
TARGETS = [{"kind": "served"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "portfolio_profile_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_profile_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_profile_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_profile_golden"

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is a synthetic two asset blend and a 2000 path simulation with a fixed seed, so no query runs.

In [ ]:
from math import sqrt

import numpy as np

YEARS = 30
PATHS = 2000


def mix(share):
    weight = share / 100.0
    variance = (weight * 15.0) ** 2 + ((1.0 - weight) * 5.0) ** 2 + 2.0 * 0.2 * weight * (1.0 - weight) * 15.0 * 5.0
    return 2.0 + 4.0 * weight, sqrt(variance)


def shortfall(share, rate):
    mean, volatility = mix(share)
    returns = np.random.default_rng(0).normal(mean / 100.0, volatility / 100.0, (PATHS, YEARS))
    wealth = np.full(PATHS, 100.0)
    for year in range(YEARS):
        wealth = (wealth - rate) * (1.0 + returns[:, year])
        wealth = np.where(wealth > 0.0, wealth, 0.0)
    return float(np.mean(wealth <= 0.0) * 100.0)


def labelled(shares, rates):
    pairs = [(share, rate) for share in shares for rate in rates]
    return {
        INPUTS[0]: [share for share, _ in pairs],
        INPUTS[1]: [rate for _, rate in pairs],
        OUTPUTS[0]: [round(mix(share)[0], 6) for share, _ in pairs],
        OUTPUTS[1]: [round(mix(share)[1], 6) for share, _ in pairs],
        OUTPUTS[2]: [round(shortfall(share, rate), 4) for share, rate in pairs],
    }


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([10.0, 30.0, 50.0, 70.0, 90.0], [2.0, 3.0, 4.0, 5.0, 6.0, 7.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([20.0, 40.0, 60.0, 80.0], [2.5, 3.5, 4.5, 5.5, 6.5]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([15.0, 35.0, 55.0, 75.0, 95.0], [3.25, 4.25, 5.25, 6.25]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit the return as a line and the variance as a quadratic in the risky share, then a logistic curve from the withdrawal rate to the shortfall probability whose midpoint moves with the mix's return and volatility, on `training` only (R6).

In [ ]:
from scipy.optimize import curve_fit

weights = training_pl[INPUTS[0]].to_numpy() / 100.0
rates = training_pl[INPUTS[1]].to_numpy()
means = training_pl[OUTPUTS[0]].to_numpy()
volatilities = training_pl[OUTPUTS[1]].to_numpy()
return_slope, return_intercept = np.polyfit(weights, means, 1)
variance_square, variance_linear, variance_constant = np.polyfit(weights, volatilities**2, 2)
fitted = curve_fit(
    lambda data, slope, m0, m1, m2: 100.0 / (1.0 + np.exp(np.minimum(-slope * (data[0] - (m0 + m1 * data[1] - m2 * data[2])), 50.0))),
    (rates, means, volatilities),
    training_pl[OUTPUTS[2]].to_numpy(),
    p0=[1.0, 3.0, 1.0, 0.05],
    maxfev=20000,
)[0]
PARAMS = {
    "return_intercept": round(float(return_intercept), 4),
    "return_slope": round(float(return_slope), 4),
    "variance_square": round(float(variance_square), 4),
    "variance_linear": round(float(variance_linear), 4),
    "variance_constant": round(float(variance_constant), 4),
    "slope": round(float(fitted[0]), 4),
    "m0": round(float(fitted[1]), 4),
    "m1": round(float(fitted[2]), 4),
    "m2": round(float(fitted[3]), 4),
}
print("fitted", PARAMS)
assert PARAMS["return_intercept"] == 2.0 and PARAMS["return_slope"] == 4.0, "the fit did not recover the asset returns"
assert PARAMS["slope"] > 0.0, "the shortfall probability does not rise with the withdrawal rate"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, run it through the library runtime and check the `validation` and `backtest` error against the limits the notebook owns (R4, R14).

In [ ]:
from workbench.model import evaluate_acceptance, measure
from workbench.replay import model_predict

MODEL_TEMPLATE = """from math import exp, sqrt

import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT_SHARE = {inputs[0]!r}
INPUT_RATE = {inputs[1]!r}
OUTPUT_RETURN = {outputs[0]!r}
OUTPUT_VOLATILITY = {outputs[1]!r}
OUTPUT_SHORTFALL = {outputs[2]!r}
RETURN_INTERCEPT = {return_intercept!r}
RETURN_SLOPE = {return_slope!r}
VARIANCE_SQUARE = {variance_square!r}
VARIANCE_LINEAR = {variance_linear!r}
VARIANCE_CONSTANT = {variance_constant!r}
SLOPE = {slope!r}
M0 = {m0!r}
M1 = {m1!r}
M2 = {m2!r}


def profile(share, rate):
    weight = share / 100.0
    mean = RETURN_INTERCEPT + RETURN_SLOPE * weight
    volatility = sqrt(max(VARIANCE_SQUARE * weight**2 + VARIANCE_LINEAR * weight + VARIANCE_CONSTANT, 0.0))
    shortfall = 100.0 / (1.0 + exp(min(-SLOPE * (rate - (M0 + M1 * mean - M2 * volatility)), 50.0)))
    return mean, volatility, shortfall


class Profile(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        results = [profile(share, rate) for share, rate in zip(model_input[INPUT_SHARE], model_input[INPUT_RATE], strict=True)]
        return pd.DataFrame(
            {{
                OUTPUT_RETURN: [result[0] for result in results],
                OUTPUT_VOLATILITY: [result[1] for result in results],
                OUTPUT_SHORTFALL: [result[2] for result in results],
            }}
        )


set_model(Profile())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS, inputs=INPUTS, outputs=OUTPUTS))

predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMITS = {OUTPUTS[0]: 0.05, OUTPUTS[1]: 0.05, OUTPUTS[2]: 10.0}
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({name: frame[name].to_list() for name in INPUTS}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMITS[output], "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows and the project from the loaded signature, verify it and replay the golden through the library loader (R1, R3, R14).

In [ ]:
def golden_samples():
    shares = [0.0, 25.0, 50.0, 75.0, 100.0]
    rates = [2.0, 3.0, 4.0, 5.0, 6.0]
    answers = predict({INPUTS[0]: shares, INPUTS[1]: rates})
    return {
        INPUTS[0]: [*shares, None],
        INPUTS[1]: [*rates, rates[0]],
        **{output: [*answers[output], None] for output in OUTPUTS},
        "expect": ["ok"] * len(shares) + ["error"],
    }

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic two asset blend and seeded simulation, training split",
        [training_path, golden_path],
        "line and quadratic for the mix, logistic curve for the shortfall",
        implementation="mlflow.pyfunc model.py, requires mlflow, pandas and the standard library",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across the range of the inputs",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)